# Yapperroni — fine-tune whisper turbo for lectures and meetings

One notebook for **Kaggle and Colab**. Everything that matters between sessions
(restored transcripts, checkpoints, test clips) lives in your private Hugging
Face repo, so a run started on one platform continues on the other.

**Before the first run**
1. Hugging Face: create a *write* token at huggingface.co/settings/tokens.
2. Optional, Weights & Biases: copy your API key from wandb.ai/authorize.
3. Add them as secrets named `HF_TOKEN` and `WANDB_API_KEY`
   — Kaggle: *Add-ons → Secrets*. Colab: the key icon in the left bar.
4. Kaggle: *Settings → Accelerator: GPU T4 x2* and *Internet: on*. Colab: *Runtime → Change runtime type → T4 GPU*.
5. Set `HF_USER` in the next cell.

**Order of work** — each step stops by itself before the session limit and
continues where it left off when you run it again, on either platform:
1. `prepare.py` — one pass of base turbo to repair transcript style (AMI is ALL CAPS, VoxPopuli has no punctuation).
2. `train.py` — LoRA training. Step 0 scores unmodified turbo: that is the number to beat.

**Memory on a free T4 (16 GB)** — starting points, not measurements: turbo in fp16 ≈ 1.6 GB,
LoRA r=32 adds ~100 MB of trainable weights, gradient checkpointing is on. If you hit
out-of-memory, halve `--batch` and double `--accum`. The first hour's log line
`s-audio/s` is the real speed; everything else is an estimate until then.

Only public datasets are used here. Training on your own recordings belongs in a separate,
clearly consented step — it is deliberately not in this notebook.

In [ ]:
HF_USER = "your-hf-username"          # <- change this
STORE = f"hf:{HF_USER}/yapperroni-train"
WANDB_PROJECT = "yapperroni"          # "" to skip Weights & Biases

import os
def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient          # Kaggle
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata                     # Colab
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

os.environ["HF_TOKEN"] = secret("HF_TOKEN") or ""
if WANDB_PROJECT:
    os.environ["WANDB_API_KEY"] = secret("WANDB_API_KEY") or ""
WANDB_ARG = f"--wandb {WANDB_PROJECT}" if WANDB_PROJECT and os.environ.get("WANDB_API_KEY") else ""
assert os.environ["HF_TOKEN"], "add an HF_TOKEN secret (a write token) first"
platform = "kaggle" if os.path.exists("/kaggle") else "colab" if "COLAB_RELEASE_TAG" in os.environ else "local"
print("platform:", platform, "| store:", STORE, "| wandb:", bool(WANDB_ARG))

In [ ]:
%%writefile text.py
"""Text side of training: scoring, and repairing the style of human transcripts.

Most free human transcripts are the right words in the wrong style. AMI and
EdAcc are ALL CAPS with no punctuation ("IF YOU IF YOU S. S. H."); VoxPopuli
is cased but unpunctuated. Training on them as-is teaches turbo to write that
way. `restore_style` keeps the human's words and borrows casing and
punctuation from what the base model itself produced for the same audio.

Run `python text.py` for the self-check.
"""
import difflib
import re

FILLERS = {"uh", "um", "er", "erm", "ah", "hmm", "mm", "mhm", "uh-huh", "huh"}
NUMBER_WORDS = set(
    "zero one two three four five six seven eight nine ten eleven twelve thirteen "
    "fourteen fifteen sixteen seventeen eighteen nineteen twenty thirty forty fifty "
    "sixty seventy eighty ninety hundred thousand million billion first second third".split())
END = ".?!"


def norm_word(w: str) -> str:
    return re.sub(r"[^a-z0-9']", "", w.lower()).strip("'")


def scoring_words(s: str) -> list[str]:
    """The normalizer every number in this project was scored with: case,
    punctuation, fillers and numbers (digits and words) ignored, so "10"
    against "ten" is not an error."""
    s = re.sub(r"[^a-z' ]", " ", s.lower().replace("-", " ").replace("%", " percent "))
    return [w.strip("'") for w in s.split()
            if w.strip("'") and w not in FILLERS and w not in NUMBER_WORDS]


def edit_distance(r: list[str], h: list[str]) -> int:
    d = list(range(len(h) + 1))
    for i in range(1, len(r) + 1):
        prev, d[0] = d[0], i
        for j in range(1, len(h) + 1):
            cur = min(d[j] + 1, d[j - 1] + 1, prev + (r[i - 1] != h[j - 1]))
            prev, d[j] = d[j], cur
    return d[len(h)]


def wer(refs: list[str], hyps: list[str]) -> float:
    errs = words = 0
    for r, h in zip(refs, hyps):
        rw = scoring_words(r)
        errs += edit_distance(rw, scoring_words(h))
        words += len(rw)
    return errs / max(1, words)


def style_stats(hyps: list[str]) -> dict:
    """What WER cannot see: training on unpunctuated text makes the model
    stop punctuating, and the normalizer above strips punctuation before
    scoring. Tracked separately so that regression shows up."""
    long = [h.strip() for h in hyps if len(h.split()) >= 4]
    if not long:
        return {"punctuated": None, "cased": None}
    return {"punctuated": sum(h[-1] in END for h in long) / len(long),
            "cased": sum(any(c.isupper() for c in h) for h in long) / len(long)}


def _sentence_case(words: list[str]) -> list[str]:
    out, start = [], True
    for w in words:
        if w == "i" or w.startswith("i'"):
            w = "I" + w[1:]
        if start and w:
            w = w[0].upper() + w[1:]
        out.append(w)
        start = w[-1:] in END if w else start
    return out


def is_shouting(text: str) -> bool:
    letters = [c for c in text if c.isalpha()]
    return bool(letters) and sum(c.isupper() for c in letters) / len(letters) > 0.9


def restore_style(human: str, teacher: str | None) -> str:
    """Human words, teacher style.

    Where the two agree on a word, the teacher's spelling of it wins — its
    casing and the punctuation attached to it. Where they disagree, the
    human word wins, lower-cased if the human text was shouting. Words only
    the teacher heard are dropped: the human transcript is the ground truth.
    Fillers are dropped too; turbo does not write them and notes should not
    have them.

    `teacher` None means "too short to bother running the model": one or two
    words get sentence case and a full stop, which is what turbo writes.
    """
    shouting = is_shouting(human)
    hw = [w for w in human.split() if norm_word(w) not in FILLERS and norm_word(w)]
    if not hw:
        return ""
    if teacher is None:
        words = [w.lower() for w in hw] if shouting else hw
        out = " ".join(_sentence_case(words))
        return out if out[-1] in END else out + "."

    tw = teacher.split()
    hn = [norm_word(w) for w in hw]
    tn = [norm_word(w) for w in tw]
    out: list[str] = []
    for op, i1, i2, j1, j2 in difflib.SequenceMatcher(a=hn, b=tn, autojunk=False).get_opcodes():
        if op == "equal":
            out += tw[j1:j2]
        elif op in ("replace", "delete"):
            out += [w.lower() if shouting else w for w in hw[i1:i2]]
    if not out:
        return ""
    if shouting:
        # Shouted words that the teacher did not vouch for need casing from
        # somewhere: sentence starts and "I". Teacher-matched words keep theirs.
        out = _sentence_case(out)
    else:
        out[0] = out[0][0].upper() + out[0][1:]
    if out[-1][-1] not in END and tw and tw[-1][-1:] in END:
        out[-1] += tw[-1][-1]
    return " ".join(out)


def _selftest() -> None:
    fails = 0

    def check(what, got, want):
        nonlocal fails
        ok = got == want
        fails += not ok
        print(f"  {'ok  ' if ok else 'FAIL'} {what}" + ("" if ok else f"\n        got  {got!r}\n        want {want!r}"))

    print("restore_style:")
    check("shouted AMI, teacher agrees",
          restore_style("IT IS NOT SPECIFICALLY POINTED AT CHINA",
                        "It is not specifically pointed at China."),
          "It is not specifically pointed at China.")
    check("human word wins where they differ",
          restore_style("WE NEED THE PYRUVATE NOW", "We need the pirouette now."),
          "We need the pyruvate now.")
    check("teacher-only words are dropped",
          restore_style("THE MEETING IS OVER", "The meeting is over. Thank you."),
          "The meeting is over.")
    check("fillers go, teacher punctuation stays",
          restore_style("UM SO I THINK UH WE SHOULD", "So, I think we should."),
          "So, I think we should.")
    check("cased but unpunctuated VoxPopuli keeps its own case",
          restore_style("Nevertheless reaching a good deal in Copenhagen is our common interest",
                        "Nevertheless, reaching a good deal in Copenhagen is our common interest."),
          "Nevertheless, reaching a good deal in Copenhagen is our common interest.")
    check("short utterance without a teacher", restore_style("YEAH", None), "Yeah.")
    check("I is capitalised", restore_style("I'M NOT SURE", None), "I'm not sure.")
    check("only fillers", restore_style("UM UH", None), "")

    print("scoring:")
    check("case, punctuation and numbers ignored",
          wer(["We have 10 essays."], ["we have ten essays"]), 0.0)
    check("one substitution in four words", wer(["a b c d"], ["a b x d"]), 0.25)
    check("style stats", style_stats(["This is a sentence.", "this is not punctuated"]),
          {"punctuated": 0.5, "cased": 0.5})

    print("PASS" if not fails else f"FAIL: {fails} case(s)")
    raise SystemExit(1 if fails else 0)


if __name__ == "__main__":
    _selftest()


In [ ]:
%%writefile data.py
"""Public speech data, streamed — nothing is downloaded whole.

Every source is read from the Hugging Face Parquet mirror of the dataset,
which needs no loading script and no account. Audio is decoded here with
soundfile rather than by `datasets`, which keeps its own decoder dependency
(and its version churn) out of the picture.
"""
import io
from dataclasses import dataclass, field

import numpy as np
import soundfile as sf
from datasets import Audio, interleave_datasets, load_dataset

SR = 16_000
MAX_SECONDS = 30.0  # whisper's window; longer clips would be truncated


@dataclass
class Source:
    repo: str
    config: str
    split: str
    text: str
    lang: str
    # Restored targets are looked up by this key. AMI's close-talk and
    # far-field copies of an utterance share one, so the teacher runs once on
    # the clean headset audio and both copies train on the result.
    key: tuple[str, ...] = ("audio_id",)
    restore: bool = True
    where: dict = field(default_factory=dict)  # column -> required value

    def url(self) -> str:
        return f"hf://datasets/{self.repo}@refs%2Fconvert%2Fparquet/{self.config}/{self.split}/*.parquet"


AMI_KEY = ("meeting_id", "speaker_id", "begin_time", "end_time")

TRAIN = {
    "ami_ihm": Source("edinburghcstr/ami", "ihm", "train", "text", "en", AMI_KEY),
    "ami_sdm": Source("edinburghcstr/ami", "sdm", "train", "text", "en", AMI_KEY),
    "vox_en": Source("facebook/voxpopuli", "en", "train", "raw_text", "en"),
    # A little French so an English-only mix does not wear away the language
    # detection that made turbo usable on French lectures.
    "vox_fr": Source("facebook/voxpopuli", "fr", "train", "raw_text", "fr"),
}
# Which source's restored targets a source trains on.
TARGETS_FROM = {"ami_sdm": "ami_ihm"}
DEFAULT_MIX = {"ami_sdm": 0.35, "ami_ihm": 0.15, "vox_en": 0.40, "vox_fr": 0.10}

# Never trained on. Fixed before the first step so every checkpoint is
# scored on the same clips.
EVAL = {
    "vox_french_accent": Source("facebook/voxpopuli", "en_accented", "test", "raw_text", "en",
                                where={"accent": "en_fr"}, restore=False),
    "ami_far_field": Source("edinburghcstr/ami", "sdm", "test", "text", "en", AMI_KEY, restore=False),
    "earnings22_calls": Source("distil-whisper/earnings22", "chunked", "test", "transcription", "en",
                               ("file_id", "segment_id"), restore=False),
}


def key_of(row: dict, src: Source) -> str:
    return "|".join(str(row[k]) for k in src.key)


def decode(audio: dict) -> np.ndarray | None:
    x, sr = sf.read(io.BytesIO(audio["bytes"]), dtype="float32", always_2d=True)
    x = x.mean(axis=1)
    if sr != SR:
        from scipy.signal import resample_poly
        g = np.gcd(sr, SR)
        x = resample_poly(x, SR // g, sr // g).astype(np.float32)
    if len(x) < SR * 0.3 or len(x) > SR * MAX_SECONDS:
        return None
    return x


def stream(src: Source, local_files: list[str] | None = None):
    """Rows with raw audio bytes. `local_files` swaps the remote Parquet for
    local ones — the smoke test runs on a few hundred rows this way."""
    ds = load_dataset("parquet", data_files=local_files or src.url(), split="train", streaming=True)
    ds = ds.cast_column("audio", Audio(decode=False))
    for col, val in src.where.items():
        ds = ds.filter(lambda r, c=col, v=val: r[c] == v)
    return ds


def mixed(mix: dict[str, float], seed: int, local: dict[str, list[str]] | None = None):
    """One stream drawing from each source in proportion. Every row carries
    its source name, so the batch knows which language prefix and which
    restored-target table it needs."""
    names = list(mix)
    parts = []
    for n in names:
        src = TRAIN[n]
        cols = ["audio", src.text, *src.key]
        ds = stream(src, (local or {}).get(n)).select_columns(cols)
        ds = ds.map(lambda r, n=n, s=src: {"source": n, "key": key_of(r, s), "human": r[s.text]},
                    remove_columns=[c for c in cols if c != "audio"])
        parts.append(ds.shuffle(seed=seed, buffer_size=500))
    total = sum(mix.values())
    return interleave_datasets(parts, probabilities=[mix[n] / total for n in names], seed=seed,
                               stopping_strategy="all_exhausted")


def eval_rows(name: str, n: int, local_files: list[str] | None = None) -> list[dict]:
    src = EVAL[name]
    out = []
    for r in stream(src, local_files):
        x = decode(r["audio"])
        if x is None or not str(r[src.text]).strip():
            continue
        out.append({"audio": x, "text": r[src.text], "lang": src.lang})
        if len(out) >= n:
            break
    return out


In [ ]:
%%writefile model.py
"""Whisper, LoRA, batching — shared by prepare.py, train.py and export."""
import numpy as np
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor

BASE = "openai/whisper-large-v3-turbo"
# Attention and MLP in both halves. The encoder is where accents and rooms
# live; leaving it frozen (--decoder-only) is cheaper and keeps the CoreML
# encoder valid, but only adapts vocabulary and style.
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"]


def device_and_dtype(prefer: str = "auto"):
    if prefer == "cpu":
        return torch.device("cpu"), torch.float32
    if torch.cuda.is_available():
        # T4 and P100 have no bf16; fp16 with a GradScaler is the path there.
        bf16 = torch.cuda.is_bf16_supported()
        return torch.device("cuda"), (torch.bfloat16 if bf16 else torch.float16)
    if torch.backends.mps.is_available():
        return torch.device("mps"), torch.float32
    return torch.device("cpu"), torch.float32


def load(name: str, device, dtype):
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()  # generate() warns on every batch otherwise
    processor = WhisperProcessor.from_pretrained(name)
    model = WhisperForConditionalGeneration.from_pretrained(name, dtype=dtype).to(device)
    model.config.use_cache = False
    return processor, model


def add_lora(model, r: int, decoder_only: bool):
    from peft import LoraConfig, get_peft_model
    targets = LORA_TARGETS
    if decoder_only:
        targets = r"model\.decoder\..*\.(" + "|".join(LORA_TARGETS) + r")"
    cfg = LoraConfig(r=r, lora_alpha=2 * r, lora_dropout=0.05, target_modules=targets, bias="none")
    return get_peft_model(model, cfg)


def features(processor, audios: list[np.ndarray], device, dtype) -> torch.Tensor:
    f = processor.feature_extractor(audios, sampling_rate=16_000, return_tensors="pt").input_features
    return f.to(device=device, dtype=dtype)


def labels(processor, texts: list[str], langs: list[str]) -> torch.Tensor:
    """Token ids as whisper is prompted at inference: start, language,
    transcribe, no-timestamps, text, end. The model prepends the start token
    itself when shifting, so it is stripped here."""
    tok = processor.tokenizer
    rows = []
    for text, lang in zip(texts, langs):
        tok.set_prefix_tokens(language=lang, task="transcribe", predict_timestamps=False)
        ids = tok(text).input_ids
        if ids and ids[0] == tok.convert_tokens_to_ids("<|startoftranscript|>"):
            ids = ids[1:]
        rows.append(ids[:440])
    width = max(len(r) for r in rows)
    out = torch.full((len(rows), width), -100, dtype=torch.long)
    for i, r in enumerate(rows):
        out[i, : len(r)] = torch.tensor(r)
    return out


@torch.no_grad()
def transcribe(model, processor, audios: list[np.ndarray], lang: str, device, dtype,
               batch: int = 16) -> list[str]:
    """Greedy, no timestamps — the same decoding the app runs."""
    was_training = model.training
    model.eval()
    out = []
    for i in range(0, len(audios), batch):
        feats = features(processor, audios[i:i + batch], device, dtype)
        ids = model.generate(input_features=feats, language=lang, task="transcribe",
                             return_timestamps=False, max_new_tokens=440)
        out += [t.strip() for t in processor.batch_decode(ids, skip_special_tokens=True)]
    if was_training:
        model.train()
    return out


In [ ]:
%%writefile store.py
"""Where checkpoints and restored transcripts live between sessions.

Option A hands one training run back and forth between Kaggle and Colab, so
nothing may live only on the machine that happens to be running: every
checkpoint and every batch of restored targets goes to the store, and every
session starts by pulling from it.

`--store runs/local` is a folder (the smoke test uses this). `--store
hf:you/yapperroni-train` is a private Hugging Face model repo; it needs
HF_TOKEN in the environment.
"""
import os
import shutil
from pathlib import Path


class Store:
    def __init__(self, spec: str, cache: str = ".cache/store"):
        self.hf = spec.startswith("hf:")
        self.cache = Path(cache)
        if self.hf:
            from huggingface_hub import HfApi
            self.repo = spec[3:]
            self.api = HfApi(token=os.environ.get("HF_TOKEN"))
            self.api.create_repo(self.repo, private=True, exist_ok=True)
            self.root = self.cache / self.repo.replace("/", "__")
        else:
            self.root = Path(spec)
        self.root.mkdir(parents=True, exist_ok=True)

    def path(self, sub: str) -> Path:
        return self.root / sub

    def push(self, sub: str) -> None:
        """Upload `sub` (a file or folder under the local root)."""
        if not self.hf:
            return
        p = self.path(sub)
        if p.is_dir():
            self.api.upload_folder(repo_id=self.repo, folder_path=str(p), path_in_repo=sub,
                                   commit_message=f"update {sub}", delete_patterns=["*"])
        else:
            self.api.upload_file(repo_id=self.repo, path_or_fileobj=str(p), path_in_repo=sub,
                                 commit_message=f"update {sub}")

    def pull(self, sub: str) -> bool:
        """Fetch `sub` from the remote. True when it exists locally afterwards."""
        if self.hf:
            from huggingface_hub import snapshot_download
            snapshot_download(self.repo, allow_patterns=[sub, f"{sub}/*"], local_dir=str(self.root),
                              token=os.environ.get("HF_TOKEN"))
        return self.path(sub).exists()

    def replace_dir(self, sub: str, staged: Path) -> None:
        """Swap a freshly written folder into place, then push it. Writing to
        a staging folder first means a crash mid-save never leaves a
        half-written checkpoint where resume will look for it."""
        dest = self.path(sub)
        if dest.exists():
            shutil.rmtree(dest)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(staged), str(dest))
        self.push(sub)


In [ ]:
%%writefile prepare.py
"""One pass of base turbo over the training data, to repair transcript style.

For each source it streams the audio, runs the base model, and writes
`targets/<source>/part-NNNNN.parquet` rows of (key, text): the human words in
turbo's casing and punctuation (see text.restore_style). Progress is saved
to the store every `--flush` rows with the stream position, so the pass can
stop on one platform and continue on another without redoing work.

    python prepare.py --store hf:you/yapperroni-train --sources ami_ihm,vox_en,vox_fr
"""
import argparse
import json
import os
import signal
import time

import pyarrow as pa
import pyarrow.parquet as pq

import data
import model as M
import text
from store import Store


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--store", required=True)
    ap.add_argument("--sources", default="ami_ihm,vox_en,vox_fr")
    ap.add_argument("--model", default=M.BASE)
    ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--flush", type=int, default=2000, help="rows per saved part")
    ap.add_argument("--max-rows", default="vox_en=72000,vox_fr=18000",
                    help="per-source caps, e.g. vox_en=72000 (~200 h)")
    ap.add_argument("--max-hours", type=float, default=11.5)
    ap.add_argument("--device", default="auto")
    ap.add_argument("--local", default="", help="source=file.parquet,... (smoke test)")
    a = ap.parse_args()

    store = Store(a.store)
    caps = {k: int(v) for k, v in (p.split("=") for p in a.max_rows.split(",") if p)}
    local = {k: [v] for k, v in (p.split("=") for p in a.local.split(",") if p)}
    device, dtype = M.device_and_dtype(a.device)
    processor, model = M.load(a.model, device, dtype)
    deadline = time.time() + a.max_hours * 3600
    stop = {"now": False}
    signal.signal(signal.SIGTERM, lambda *_: stop.update(now=True))

    for name in a.sources.split(","):
        src = data.TRAIN[name]
        sub = f"targets/{name}"
        store.pull(sub)
        state_file = store.path(f"{sub}/state.json")
        state = json.loads(state_file.read_text()) if state_file.exists() else {"rows": 0, "part": 0, "done": False}
        if state["done"]:
            print(f"{name}: already complete ({state['rows']} rows)")
            continue
        ds = data.stream(src, local.get(name))
        if "stream" in state:
            ds.load_state_dict(state["stream"])
        print(f"{name}: resuming at {state['rows']} rows" if state["rows"] else f"{name}: starting")

        rows, queue, t0, seen_keys = [], [], time.time(), set()

        def run_queue():
            if not queue:
                return
            hyps = M.transcribe(model, processor, [q[2] for q in queue], src.lang, device, dtype, a.batch)
            for (k, human, _), hyp in zip(queue, hyps):
                rows.append({"key": k, "text": text.restore_style(human, hyp)})
            queue.clear()

        def flush(final=False):
            run_queue()
            if rows:
                p = store.path(f"{sub}/part-{state['part']:05d}.parquet")
                p.parent.mkdir(parents=True, exist_ok=True)
                pq.write_table(pa.Table.from_pylist(rows), p)
                state["part"] += 1
                state["rows"] += len(rows)
                rows.clear()
            state["stream"] = ds.state_dict()
            state["done"] = final
            state_file.parent.mkdir(parents=True, exist_ok=True)
            state_file.write_text(json.dumps(state))
            store.push(sub)
            rate = state["rows"] / max(1e-6, time.time() - t0)
            print(f"{name}: {state['rows']} rows saved ({rate:.1f} rows/s this session)", flush=True)

        exhausted = True
        for r in ds:
            human = str(r[src.text] or "").strip()
            k = data.key_of(r, src)
            if not human or k in seen_keys:
                continue
            seen_keys.add(k)
            x = data.decode(r["audio"])
            if x is None:
                continue
            content = [w for w in human.split() if text.norm_word(w) not in text.FILLERS]
            if len(content) <= 2:
                rows.append({"key": k, "text": text.restore_style(human, None)})
            else:
                queue.append((k, human, x))
                if len(queue) >= a.batch:
                    run_queue()
            if len(rows) >= a.flush:
                flush()
            if state["rows"] + len(rows) + len(queue) >= caps.get(name, 10**12):
                break
            if stop["now"] or time.time() > deadline:
                exhausted = False
                break
        flush(final=exhausted)
        if not exhausted:
            print("stopping: time limit or SIGTERM — rerun to continue")
            break
    # The streaming readers' cleanup can hang interpreter shutdown.
    os._exit(0)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile train.py
"""LoRA fine-tune of whisper turbo, resumable across Kaggle and Colab sessions.

Every `--save-minutes` and before `--max-hours` runs out, the adapter, the
optimizer, the schedule and the exact stream position go to the store; a new
session on either platform pulls the latest and carries on from the same
step and the same row. Eval runs at step 0 — the number to beat — and every
`--eval-every` steps on clips that are never trained on.

    python train.py --store hf:you/yapperroni-train
"""
import argparse
import contextlib
import json
import os
import random
import signal
import tempfile
import time
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq
import torch

import data
import model as M
import text
from store import Store


def load_targets(store: Store, names: set[str]) -> dict[str, dict[str, str]]:
    out = {}
    for n in names:
        store.pull(f"targets/{n}")
        parts = sorted(store.path(f"targets/{n}").glob("part-*.parquet"))
        table = {}
        for p in parts:
            t = pq.read_table(p).to_pydict()
            table.update(zip(t["key"], t["text"]))
        out[n] = table
        print(f"targets {n}: {len(table)} restored transcripts")
    return out


def load_eval(store: Store, names: list[str], n: int, local: dict) -> dict[str, list[dict]]:
    """Cached in the store after the first build, so every session scores the
    very same clips and none of them re-streams a 2 GB shard to find them."""
    sets = {}
    for name in names:
        sub = f"eval/{name}-{n}.npz"
        if store.pull(sub):
            # Plain arrays, no pickle: the cache comes back from a remote repo.
            z = np.load(store.path(sub), allow_pickle=False)
            cuts = np.cumsum(z["lengths"])[:-1]
            sets[name] = [{"audio": x, "text": str(t), "lang": str(z["lang"])}
                          for x, t in zip(np.split(z["audio"], cuts), z["texts"])]
        else:
            rows = data.eval_rows(name, n, local.get(name))
            store.path("eval").mkdir(parents=True, exist_ok=True)
            np.savez(store.path(sub), audio=np.concatenate([r["audio"] for r in rows]),
                     lengths=np.array([len(r["audio"]) for r in rows]),
                     texts=np.array([r["text"] for r in rows]), lang=np.array(rows[0]["lang"]))
            store.push(sub)
            sets[name] = rows
        print(f"eval {name}: {len(sets[name])} clips")
    return sets


def evaluate(model, processor, sets, device, dtype, amp) -> dict:
    res = {}
    with amp():
        for name, rows in sets.items():
            hyps = M.transcribe(model, processor, [r["audio"] for r in rows], rows[0]["lang"],
                                device, dtype, batch=8)
            res[name] = {"wer": text.wer([r["text"] for r in rows], hyps), **text.style_stats(hyps)}
    res["mean_wer"] = float(np.mean([v["wer"] for v in res.values()]))
    return res


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--store", required=True)
    ap.add_argument("--model", default=M.BASE)
    ap.add_argument("--mix", default=",".join(f"{k}={v}" for k, v in data.DEFAULT_MIX.items()))
    ap.add_argument("--batch", type=int, default=8)
    ap.add_argument("--accum", type=int, default=4)
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--warmup", type=int, default=200)
    ap.add_argument("--max-steps", type=int, default=6000)
    ap.add_argument("--lora-r", type=int, default=32)
    ap.add_argument("--decoder-only", action="store_true")
    ap.add_argument("--eval-every", type=int, default=500)
    ap.add_argument("--eval-n", type=int, default=200)
    ap.add_argument("--save-minutes", type=float, default=30)
    ap.add_argument("--max-hours", type=float, default=11.5)
    ap.add_argument("--raw-targets", action="store_true",
                    help="train on unrestored transcripts (teaches ALL CAPS, no punctuation)")
    ap.add_argument("--device", default="auto")
    ap.add_argument("--seed", type=int, default=7)
    ap.add_argument("--local", default="", help="source=file.parquet,... (smoke test)")
    ap.add_argument("--wandb", default="", help="W&B project name; needs WANDB_API_KEY")
    a = ap.parse_args()

    started = time.time()
    store = Store(a.store)
    local = {k: [v] for k, v in (p.split("=") for p in a.local.split(",") if p)}
    mix = {k: float(v) for k, v in (p.split("=") for p in a.mix.split(","))}
    device, dtype = M.device_and_dtype(a.device)
    amp = (lambda: torch.autocast("cuda", dtype=dtype)) if device.type == "cuda" else contextlib.nullcontext
    print(f"device {device} {dtype}")

    processor, model = M.load(a.model, device, dtype)
    model = M.add_lora(model, a.lora_r, a.decoder_only)
    model.base_model.model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.print_trainable_parameters()
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=a.lr, weight_decay=0.01)
    from transformers import get_cosine_schedule_with_warmup
    sched = get_cosine_schedule_with_warmup(opt, a.warmup, a.max_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=(dtype == torch.float16))

    epoch, wandb_id = 0, None
    stream = data.mixed(mix, a.seed, local)
    step, best, history = 0, float("inf"), []
    if store.pull("checkpoints/latest/meta.json"):
        store.pull("checkpoints/latest")
        ck = store.path("checkpoints/latest")
        meta = json.loads((ck / "meta.json").read_text())
        from peft import set_peft_model_state_dict
        from safetensors.torch import load_file
        set_peft_model_state_dict(model, load_file(ck / "adapter" / "adapter_model.safetensors"))
        tr = torch.load(ck / "trainer.pt", map_location="cpu", weights_only=True)
        opt.load_state_dict(tr["opt"]); sched.load_state_dict(tr["sched"]); scaler.load_state_dict(tr["scaler"])
        random.setstate(tr["py_rng"]); torch.set_rng_state(tr["torch_rng"])
        epoch = meta.get("epoch", 0)
        wandb_id = meta.get("wandb_id")
        stream = data.mixed(mix, a.seed + epoch, local)
        stream.load_state_dict(meta["stream"])
        step, best, history = meta["step"], meta["best"], meta["history"]
        print(f"resumed at step {step} (best mean WER {best:.4f})")

    # One W&B run for the whole training, however many sessions and platforms
    # it takes: its id rides in the checkpoint and every session resumes it.
    wb = None
    if a.wandb:
        import wandb
        wandb_id = wandb_id or __import__("uuid").uuid4().hex[:12]
        wb = wandb.init(project=a.wandb, id=wandb_id, resume="allow", config=vars(a),
                        name=f"turbo-lora-r{a.lora_r}")

    def log(d: dict):
        if wb:
            wb.log(d, step=step)

    targets = {} if a.raw_targets else load_targets(
        store, {data.TARGETS_FROM.get(n, n) for n in mix if data.TRAIN[n].restore})
    evals = load_eval(store, list(data.EVAL), a.eval_n, local)

    def save(why: str, metrics: dict | None = None):
        nonlocal best
        stage = Path(tempfile.mkdtemp(prefix="ckpt-"))
        model.save_pretrained(stage / "adapter")
        torch.save({"opt": opt.state_dict(), "sched": sched.state_dict(), "scaler": scaler.state_dict(),
                    "py_rng": random.getstate(), "torch_rng": torch.get_rng_state()}, stage / "trainer.pt")
        improved = metrics is not None and metrics["mean_wer"] < best
        if improved:
            best = metrics["mean_wer"]
        (stage / "meta.json").write_text(json.dumps(
            {"step": step, "epoch": epoch, "best": best, "history": history, "stream": stream.state_dict(),
             "wandb_id": wandb_id,
             "args": vars(a)}, default=str))
        if improved:
            import shutil
            shutil.copytree(stage, stage.parent / (stage.name + "-best"))
            store.replace_dir("checkpoints/best", stage.parent / (stage.name + "-best"))
        store.replace_dir("checkpoints/latest", stage)
        print(f"saved step {step} ({why}){' — new best' if improved else ''}", flush=True)

    def run_eval():
        m = evaluate(model, processor, evals, device, dtype, amp)
        history.append({"step": step, **m})
        log({"eval/mean_wer": m["mean_wer"], **{f"eval/{k}/{f}": v[f] for k, v in m.items()
                                                 if isinstance(v, dict) for f in v if v[f] is not None}})
        print("eval step %d: " % step + "  ".join(
            f"{k} {v['wer']*100:.1f}% (punct {v['punctuated'] if v['punctuated'] is None else round(v['punctuated']*100)}%)"
            for k, v in m.items() if k != "mean_wer") + f"  | mean {m['mean_wer']*100:.2f}%", flush=True)
        return m

    stop = {"now": False}
    signal.signal(signal.SIGTERM, lambda *_: stop.update(now=True))
    deadline = started + a.max_hours * 3600
    last_save = time.time()

    if step == 0:
        save("baseline", run_eval())

    def rows():
        # One pass over the mix is an epoch; the next one reshuffles. The
        # epoch is part of the checkpoint so resume lands in the right pass.
        nonlocal stream, epoch
        while True:
            yield from stream
            epoch += 1
            print(f"epoch {epoch} begins", flush=True)
            stream = data.mixed(mix, a.seed + epoch, local)

    model.train()
    batch, micro, audio_secs, missing, t_log = [], 0, 0.0, 0, time.time()
    for row in rows():
        name = row["source"]
        src = data.TRAIN[name]
        if a.raw_targets or not src.restore:
            target = row["human"]
        else:
            target = targets[data.TARGETS_FROM.get(name, name)].get(row["key"])
            if target is None:
                missing += 1
                continue
        x = data.decode(row["audio"])
        if x is None or not target:
            continue
        batch.append((x, target, src.lang))
        if len(batch) < a.batch:
            continue

        feats = M.features(processor, [b[0] for b in batch], device, dtype)
        lab = M.labels(processor, [b[1] for b in batch], [b[2] for b in batch]).to(device)
        audio_secs += sum(len(b[0]) for b in batch) / data.SR
        batch = []
        with amp():
            loss = model(input_features=feats, labels=lab).loss / a.accum
        scaler.scale(loss).backward()
        micro += 1
        if micro % a.accum:
            continue

        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
        step += 1
        if step % 10 == 0:
            dt = time.time() - t_log
            log({"train/loss": loss.item() * a.accum, "train/lr": sched.get_last_lr()[0],
                 "train/audio_seconds_per_second": audio_secs / dt, "train/epoch": epoch,
                 "train/rows_without_target": missing})
            print(f"step {step} loss {loss.item() * a.accum:.4f} lr {sched.get_last_lr()[0]:.2e} "
                  f"{audio_secs / dt:.1f} s-audio/s{f' ({missing} rows had no restored target)' if missing else ''}",
                  flush=True)
            audio_secs, t_log = 0.0, time.time()

        metrics = run_eval() if step % a.eval_every == 0 else None
        if metrics or time.time() - last_save > a.save_minutes * 60:
            save("eval" if metrics else "timer", metrics)
            last_save = time.time()
        if step >= a.max_steps or stop["now"] or time.time() > deadline - 600:
            break

    if step >= a.max_steps:
        save("finished", run_eval())
    else:
        save("session limit — rerun to continue")
    if wb:
        wb.finish()
    os._exit(0)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile requirements.txt
# Training, on Kaggle / Colab / this Mac. Pinned to the set the smoke test ran
# on: the two platforms ship different preinstalled versions, and an unpinned
# install is the first thing to break when a run moves between them. torch is
# deliberately not pinned — both platforms' CUDA builds are newer than 2.4 and
# reinstalling it costs minutes per session.
torch>=2.4
transformers==5.18.0
peft==0.21.2
datasets==5.0.1
huggingface_hub==1.33.0
safetensors==0.8.0
soundfile==0.14.0
scipy==1.18.1
pyarrow==25.0.1
wandb==0.30.0


In [ ]:
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python text.py

## Step 1 — repair transcript style (run until every source says complete)
Roughly a few GPU-hours in total; it saves every 2,000 rows. Re-run this cell in a new
session to continue.

In [ ]:
!python prepare.py --store {STORE} --max-hours 11

## Step 2 — train
Re-run in each new session, on Kaggle or Colab: it resumes from the latest checkpoint
in the store. Watch `eval step …` lines (or W&B): mean WER should fall below step 0,
and `punct` should stay near 100%.

In [ ]:
!python train.py --store {STORE} --max-hours 11.5 {WANDB_ARG}